# Session 16 · Overfitting: The Deep Dive

The session the whole course has pointed at. In **Session 1** we said *a good model
learns the pattern; a bad one memorises the data.* In **Session 9** a k=1 KNN scored
**100% on its own training data**. In **Session 15** training accuracy pulled ahead of
test. Today all three come due — in one picture.

We crank a tree's `max_depth` and watch **two** accuracies: on students it trained on,
and on students it's never seen.

> ✏️ = your cell. Gaps never block the run.

## Step 1 · Setup — the students it studied vs the students it hasn't met

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split

df = pd.read_csv("../../../datasets/anchor/student_habits.csv")
habits = ["study_hours_per_week","attendance_pct","sleep_hours_per_night","screen_time_hours_per_day","practice_sessions_per_week"]
X, y = df[habits], df['passed']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print('train students:', len(y_train), '  test students:', len(y_test))

## Step 2 · ✏️ Predict BEFORE you plot

Commit a guess so the reveal lands. As `max_depth` goes from 1 to unlimited:
- What does **training** accuracy do?
- What does **test** accuracy do?

*(Write your two predictions here, replacing this line. One phrase each is fine —
e.g. 'training: ...', 'test: ...'.)*

## Step 3 · Sweep the depth — record BOTH accuracies

For every depth, fit a tree and score it twice. `max_depth=99` on the plot means
'unlimited' (grow until every leaf is pure).

In [ ]:
depths = list(range(1, 16)) + [None]        # 1..15, then unlimited
rows = []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    rows.append({'max_depth': (99 if d is None else d),   # 99 = 'unlimited' on the x-axis
                 'label': ('none' if d is None else str(d)),
                 'train_acc': t.score(X_train, y_train),
                 'test_acc':  t.score(X_test,  y_test)})
curve = pd.DataFrame(rows)

In [ ]:
print(curve[['label','train_acc','test_acc']].round(3).to_string(index=False))

## Step 4 · The reveal — two curves on one axis

Now look. One line is the students it **studied**; the other is students it's **never
met**. Watch what each does — and watch the space between them.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(curve['max_depth'], curve['train_acc'], 'o-', color='tab:red',
        label='training accuracy (students it studied)')
ax.plot(curve['max_depth'], curve['test_acc'], 'o-', color='tab:blue',
        label='test accuracy (students it has never met)')

peak = curve.loc[curve['test_acc'].idxmax()]
ax.axvline(peak['max_depth'], ls='--', color='gray', alpha=0.7)
ax.annotate('test peaks here\n-> STOP', xy=(peak['max_depth'], peak['test_acc']),
            xytext=(peak['max_depth']+2, peak['test_acc']-0.06),
            arrowprops=dict(arrowstyle='->'), fontsize=10)

ax.set_xlabel('max_depth  (99 = unlimited)')
ax.set_ylabel('accuracy')
ax.set_title('Overfitting: training climbs to 1.0, test peaks then FALLS')
ax.legend(loc='lower left')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('Training marches to a perfect 1.000.')
print('Test rises, peaks near depth 5-6 (~0.89), then FALLS to ~0.82.')
print('The fanning gap between the lines = OVERFITTING.')

## Step 5 · Where do we STOP?

The best tree is **not** the deepest (that one memorises). It's the one at the **peak
of the test curve**. Find it.

In [ ]:
best = curve.loc[curve['test_acc'].idxmax()]
deepest = curve.iloc[-1]
print(f"BEST to ship:  max_depth={best['label']:>4}  "
      f"train={best['train_acc']:.3f}  test={best['test_acc']:.3f}")
print(f"DEEPEST tree:  max_depth={deepest['label']:>4}  "
      f"train={deepest['train_acc']:.3f}  test={deepest['test_acc']:.3f}")
print()
print('The deepest tree is PERFECT on training and WORSE on new students.')
print('We ship the peak-of-test tree, not the deepest one.')

## Step 6 · Confront the memoriser

Open the unlimited tree. Its training accuracy is 1.000 — it got *every* training
student right. It did that by carving out private rules for individual students
(including noisy exceptions). Count how many tiny leaves it needed.

In [ ]:
full_tree = DecisionTreeClassifier(max_depth=None, random_state=0).fit(X_train, y_train)
n_leaves = full_tree.get_n_leaves()
print('unlimited tree — training accuracy:', round(full_tree.score(X_train, y_train), 3))
print('unlimited tree — test accuracy    :', round(full_tree.score(X_test, y_test), 3))
print('number of leaves (final answer-boxes):', n_leaves,
      'for', len(y_train), 'training students')
print()
print('So many leaves for so few students = a private rule per handful of students.')
print('That is memorising. It is also the k=1 KNN from Session 9, in tree form:')
print('both hit 100% on training by letting single points dictate the answer.')

## Step 7 · ✏️ Name it, and generalise it

Two short answers:
1. **Define overfitting in your own words** — use the phrase *memorising vs learning*
   from Session 1, and say *how you would detect it*.
2. Name **one other knob** in this course where the same danger lurks (hint: think back
   to a model from Module 3 that could score 100% on training).

*(Write both answers here, replacing this line.)*

## Wrap-up

- Training accuracy **always** improves with complexity; test accuracy **peaks then falls**.
- That widening gap is **overfitting** — memorising the training data's accidents.
- Only **held-out** evaluation catches it; training accuracy alone is blind.
- Ship the model at the **peak of the test curve**, not the most complex one.

**Next (S17):** if one deep tree memorises, grow a *forest* of them and vote — the power
of depth without the overfitting.